# Notebook to load the NuminaMath dataset

### Imports

In [1]:
import torch
try:
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"CUDA version: {torch.version.cuda}")
except Exception as e:
    print(f"Error: {e}")

PyTorch version: 2.11.0+cu130
CUDA available: True
CUDA version: 13.0


In [2]:
import os
from pathlib import Path
import torch

#------------ CACHE ENVIRONMENT ----------------------------------------
# Set cache environment variables BEFORE importing datasets
PROJECT_ROOT = Path.cwd().parent  # Go up one level
hf_cache = PROJECT_ROOT / ".cache" / "huggingface"
hf_cache.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(hf_cache)
os.environ["HF_DATASETS_CACHE"] = str(hf_cache / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache / "transformers")

#--------------------------------------- DATASET -------------------------------
HF_DATASET_ID = "AI-MO/NuminaMath-TIR"

DATASET_DIR = PROJECT_ROOT / "datasets"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

#----------------------------------------------- MODELE ------------------------
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"
# Extract model name from MODEL_ID (everything after the last /)
MODEL_NAME = MODEL_ID.split("/")[-1].lower()
MODEL_DIR = PROJECT_ROOT / "models" / MODEL_NAME

MODEL_DIR.mkdir(parents=True, exist_ok=True)

#-------------------------------------------- IMPORTS ---------------------------
# Now import the libraries
from datasets import load_dataset, load_from_disk
from pprint import pprint
from transformers import AutoModelForCausalLM, AutoTokenizer

### Loading the dataset from HF or disk

In [3]:
dataset_is_empty = not any(DATASET_DIR.iterdir())

if dataset_is_empty:
    print(f"{DATASET_DIR} is empty -> downloading from Hub and saving to disk...")
    # dataset = load_dataset(HF_DATASET_ID)
    dataset = load_dataset(HF_DATASET_ID)
    dataset.save_to_disk(str(DATASET_DIR))
else:
    print(f"{DATASET_DIR} is not empty -> loading from disk...")
    dataset = load_from_disk(str(DATASET_DIR))

/home/benjamin/Folders_Python/GenAI_Math/datasets is not empty -> loading from disk...


### Exploring the dataset

In [4]:
pprint(dataset)

DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'messages'],
        num_rows: 72441
    })
    test: Dataset({
        features: ['problem', 'solution', 'messages'],
        num_rows: 99
    })
})


In [5]:
train_dataset = dataset['train']

In [6]:
example = train_dataset[0]
pprint(example)

{'messages': [{'content': 'What is the coefficient of $x^2y^6$ in the '
                          'expansion of '
                          '$\\left(\\frac{3}{5}x-\\frac{y}{2}\\right)^8$?  '
                          'Express your answer as a common fraction.',
               'role': 'user'},
              {'content': 'To determine the coefficient of \\(x^2y^6\\) in the '
                          'expansion of \\(\\left(\\frac{3}{5}x - '
                          '\\frac{y}{2}\\right)^8\\), we can use the binomial '
                          'theorem.\n'
                          '\n'
                          'The binomial theorem states:\n'
                          '\\[\n'
                          '(a + b)^n = \\sum_{k=0}^{n} \\binom{n}{k} a^{n-k} '
                          'b^k\n'
                          '\\]\n'
                          '\n'
                          'In this case, \\(a = \\frac{3}{5}x\\), \\(b = '
                          '-\\frac{y}{2}\\), and \\(n = 8\\).

### Load model

In [7]:
model_is_empty = not any(MODEL_DIR.iterdir())

print(f"Fetching {MODEL_ID}")

if model_is_empty:
    print(f"{MODEL_DIR} is empty -> downloading model/tokenizer from Hub and saving to disk...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        device_map="auto",
    )
    tokenizer.save_pretrained(str(MODEL_DIR))
    model.save_pretrained(str(MODEL_DIR))
else:
    print(f"{MODEL_DIR} is not empty -> loading model/tokenizer from disk...")
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        str(MODEL_DIR),
        local_files_only=True,
        dtype=torch.bfloat16,
        device_map="auto",
    )

Fetching Qwen/Qwen2-0.5B-Instruct
/home/benjamin/Folders_Python/GenAI_Math/models/qwen2-0.5b-instruct is empty -> downloading model/tokenizer from Hub and saving to disk...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]